In [1]:
import os
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


In [ ]:
class BraTSDataset3D(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.cases = []

        for case in sorted(os.listdir(root_dir)):
            case_path = os.path.join(root_dir, case)
            if not os.path.isdir(case_path):
                continue

            files = os.listdir(case_path)
            if any(f.endswith("_t1.nii") for f in files) and \
               any(f.endswith("_t1ce.nii") for f in files) and \
               any(f.endswith("_seg.nii") for f in files):
                self.cases.append(case)

        print(f"✅ Loaded {len(self.cases)} valid BraTS cases")

    def __len__(self):
        return len(self.cases)

    def normalize(self, x):
        return (x - x.mean()) / (x.std() + 1e-5)

    def __getitem__(self, idx):
        case = self.cases[idx]
        path = os.path.join(self.root_dir, case)

        t1_path = [f for f in os.listdir(path) if f.endswith("_t1.nii")][0]
        t1ce_path = [f for f in os.listdir(path) if f.endswith("_t1ce.nii")][0]
        seg_path = [f for f in os.listdir(path) if f.endswith("_seg.nii")][0]

        t1 = nib.load(os.path.join(path, t1_path)).get_fdata()
        t1ce = nib.load(os.path.join(path, t1ce_path)).get_fdata()
        seg = nib.load(os.path.join(path, seg_path)).get_fdata()

        # Normalize
        t1 = self.normalize(t1)
        t1ce = self.normalize(t1ce)

        # 🔴 CONSISTENT CROPPING (divisible by 2³)
        t1 = t1[40:200, 40:200, 20:148]
        t1ce = t1ce[40:200, 40:200, 20:148]
        seg = seg[40:200, 40:200, 20:148]

        image = np.stack([t1, t1ce], axis=0)
        mask = (seg > 0).astype(np.uint8)

        return (
            torch.tensor(image, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.long)
        )


In [31]:
def center_crop(tensor, target_shape):
    _, _, d, h, w = tensor.shape
    td, th, tw = target_shape

    d1 = (d - td) // 2
    h1 = (h - th) // 2
    w1 = (w - tw) // 2

    return tensor[:, :, d1:d1+td, h1:h1+th, w1:w1+tw]


In [32]:
#🔹 Block-3: 3D U-Net Model
class UNet3D(nn.Module):
    def __init__(self):
        super().__init__()

        self.d1 = DoubleConv(2, 32)
        self.d2 = DoubleConv(32, 64)
        self.d3 = DoubleConv(64, 128)

        self.pool = nn.MaxPool3d(2)

        self.bottleneck = DoubleConv(128, 256)

        self.up3 = nn.ConvTranspose3d(256, 128, 2, 2)
        self.u3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose3d(128, 64, 2, 2)
        self.u2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose3d(64, 32, 2, 2)
        self.u1 = DoubleConv(64, 32)

        self.out = nn.Conv3d(32, 2, 1)

    def forward(self, x):
        x1 = self.d1(x)
        x2 = self.d2(self.pool(x1))
        x3 = self.d3(self.pool(x2))

        x4 = self.bottleneck(self.pool(x3))

        x = self.up3(x4)
        x3 = center_crop(x3, x.shape[2:])
        x = self.u3(torch.cat([x, x3], dim=1))

        x = self.up2(x)
        x2 = center_crop(x2, x.shape[2:])
        x = self.u2(torch.cat([x, x2], dim=1))

        x = self.up1(x)
        x1 = center_crop(x1, x.shape[2:])
        x = self.u1(torch.cat([x, x1], dim=1))

        return self.out(x)


In [33]:
#🔹 Block-4: Dice Loss
def dice_loss(pred, target):
    pred = torch.softmax(pred, dim=1)[:, 1]  # (B, D, H, W)

    # 🔴 Crop target to match pred
    if pred.shape != target.shape:
        target = center_crop(target.unsqueeze(1), pred.shape[1:]).squeeze(1)

    target = target.float()

    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()

    return 1 - (2 * intersection + 1) / (union + 1)



In [34]:
train_dataset = BraTSDataset3D(
    root_dir="BraTS2020_TrainingData"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0
)


In [35]:
val_dataset = BraTSDataset3D(
    root_dir="BraTS2020_ValidationData"
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0
)


In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet3D().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [37]:
from tqdm import tqdm
import torch

num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=False)

    for x, y in loop:
        x = x.to(device, non_blocking=True)   # (B, 2, D, H, W)
        y = y.to(device, non_blocking=True)   # (B, D, H, W)

        optimizer.zero_grad(set_to_none=True)

        pred = model(x)
        loss = dice_loss(pred, y)

        # Safety check
        if torch.isnan(loss) or torch.isinf(loss):
            print("⚠️ Skipping batch due to invalid loss")
            continue

        loss.backward()

        # IMPORTANT for 3D networks
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Training Dice Loss: {avg_loss:.4f}")


Epoch [1/20]:   0%|          | 0/30 [00:00<?, ?it/s]

IndexError: list index out of range

In [ ]:
model.eval()
val_loss = 0

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        y = y.to(device)

        pred = model(x)
        loss = dice_loss(pred, y)
        val_loss += loss.item()

val_loss /= len(val_loader)
print(f"Validation Dice Loss: {val_loss:.4f}")
